In [49]:
!pip -q install nltk pandas
import re
import random
import unicodedata
import pandas as pd
from nltk.tokenize import RegexpTokenizer
from nltk.stem import SnowballStemmer

In [54]:
tokenizer = RegexpTokenizer(r'\w+')
stemmer = SnowballStemmer("spanish")

def normalizar_texto(texto):
    """
    Convierte el texto a minúsculas, elimina acentos y caracteres especiales.
    """
    texto = texto.lower()
    texto = unicodedata.normalize("NFD", texto)
    texto = texto.encode("ascii", "ignore").decode("utf-8")
    texto = re.sub(r"[^a-zñ0-9\s]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

def tokenizar_y_stem(texto):
    """
    Tokeniza el texto y aplica stemming para reducir variaciones morfológicas.
    """
    texto = normalizar_texto(texto)
    tokens = tokenizer.tokenize(texto)
    stems = [stemmer.stem(token) for token in tokens]
    return tokens, stems
mensaje_ejemplo = "¿Dónde adjunto DOCUMENTOS?"
print("Original:", mensaje_ejemplo)
print("Normalizado:", normalizar_texto(mensaje_ejemplo))
print("Token", tokenizar_y_stem(mensaje_ejemplo))

Original: ¿Dónde adjunto DOCUMENTOS?
Normalizado: donde adjunto documentos
Token (['donde', 'adjunto', 'documentos'], ['dond', 'adjunt', 'document'])
Original: ¿Dónde adjunto DOCUMENTOS?
Normalizado: donde adjunto documentos
Token (['donde', 'adjunto', 'documentos'], ['dond', 'adjunt', 'document'])


In [55]:
# Base de conocimiento del asistente de Seguros del Norte
# Consultas frecuentes, ejemplos y respuestas del asistente de Seguros del Norte.
intenciones = {
    "saludo": {
        "ejemplos": ["hola", "buenos dias", "buenas tardes", "buenas noches", "saludos"],
        "respuestas": ["Hola. Puedo orientarte sobre documentos, operaciones y uso de la plataforma."]
    },
    "ayuda": {
        "ejemplos": ["necesito ayuda", "ayudame por favor", "puedes ayudarme", "orientame"],
        "respuestas": ["Claro. Indica si tu duda es sobre documentos, operaciones o uso de la plataforma."]
    },
    "documentacion": {
        "ejemplos": ["que documentos necesito", "documentacion requerida", "requisitos del tramite", "comprobante de pago"],
        "respuestas": ["Los documentos dependen del trámite. Especifica la operación y revisa la lista de requisitos antes de continuar."]
    },
    "procedimiento": {
        "ejemplos": ["como registro una operacion", "registrar una operacion", "nuevo registro", "pasos para registrar"],
        "respuestas": ["Para registrar una operación, entra a Operaciones y selecciona Nuevo registro."]
    },
    "adjuntar": {
        "ejemplos": ["donde adjunto archivos", "como subir archivos", "adjuntar documentos", "cargar archivos"],
        "respuestas": ["Para adjuntar archivos, abre el formulario y busca el apartado Documentos."]
    },
    "ubicacion": {
        "ejemplos": ["donde encuentro operaciones", "donde esta el menu", "ubicacion del modulo", "encontrar operaciones"],
        "respuestas": ["Encontrarás Operaciones en el menú principal."]
    },
    "falla": {
        "ejemplos": ["la pagina no carga", "la plataforma falla", "error tecnico", "pantalla bloqueada"],
        "respuestas": ["Actualiza la página y vuelve a intentarlo. Si la falla persiste, contacta al responsable de soporte técnico."]
    },
    "soporte": {
        "ejemplos": ["necesito soporte tecnico", "reportar incidente tecnico", "contactar soporte", "levantar reporte tecnico"],
        "respuestas": ["El soporte humano atiende fallas técnicas. Si tu consulta es operativa, describe el procedimiento o la función que buscas."]
    },
    "despedida": {
        "ejemplos": ["gracias", "adios", "hasta luego", "eso es todo"],
        "respuestas": ["Gracias por tu consulta. Hasta pronto."]
    }
}
print("Intenciones definidas:", list(intenciones))
for nombre, datos in intenciones.items():
    print(f"{nombre}: {len(datos['ejemplos'])} ejemplos y {len(datos['respuestas'])} respuestas")


Intenciones definidas: ['saludo', 'ayuda', 'documentacion', 'procedimiento', 'adjuntar', 'ubicacion', 'falla', 'soporte', 'despedida']
saludo: 5 ejemplos y 1 respuestas
ayuda: 4 ejemplos y 1 respuestas
documentacion: 4 ejemplos y 1 respuestas
procedimiento: 4 ejemplos y 1 respuestas
adjuntar: 4 ejemplos y 1 respuestas
ubicacion: 4 ejemplos y 1 respuestas
falla: 4 ejemplos y 1 respuestas
soporte: 4 ejemplos y 1 respuestas
despedida: 4 ejemplos y 1 respuestas
Intenciones definidas: ['saludo', 'ayuda', 'documentacion', 'procedimiento', 'adjuntar', 'ubicacion', 'falla', 'soporte', 'despedida']
saludo: 5 ejemplos y 1 respuestas
ayuda: 4 ejemplos y 1 respuestas
documentacion: 4 ejemplos y 1 respuestas
procedimiento: 4 ejemplos y 1 respuestas
adjuntar: 4 ejemplos y 1 respuestas
ubicacion: 4 ejemplos y 1 respuestas
falla: 4 ejemplos y 1 respuestas
soporte: 4 ejemplos y 1 respuestas
despedida: 4 ejemplos y 1 respuestas


In [56]:
def construir_vocabulario_intenciones(intenciones):
    """
    Construye una representación basada en stems para cada intención.
    """
    vocabulario = {}

    for nombre_intencion, datos in intenciones.items():
        stems_intencion = []

        for ejemplo in datos["ejemplos"]:
            _, stems = tokenizar_y_stem(ejemplo)
            stems_intencion.extend(stems)

        vocabulario[nombre_intencion] = set(stems_intencion)

    return vocabulario

vocabulario_intenciones = construir_vocabulario_intenciones(intenciones)
for nombre, raices in vocabulario_intenciones.items():
    print(f"{nombre}: {sorted(raices)[:12]}")

saludo: ['buen', 'dias', 'hol', 'noch', 'salud', 'tard']
ayuda: ['ayud', 'ayudam', 'favor', 'necesit', 'orientam', 'por', 'pued']
documentacion: ['comprob', 'de', 'del', 'document', 'necesit', 'pag', 'que', 'requer', 'requisit', 'tramit']
procedimiento: ['com', 'nuev', 'oper', 'par', 'pas', 'registr', 'una']
adjuntar: ['adjunt', 'archiv', 'carg', 'com', 'document', 'dond', 'sub']
ubicacion: ['del', 'dond', 'el', 'encontr', 'encuentr', 'esta', 'menu', 'modul', 'oper', 'ubic']
falla: ['bloqu', 'carg', 'error', 'fall', 'la', 'no', 'pagin', 'pantall', 'plataform', 'tecnic']
soporte: ['contact', 'incident', 'levant', 'necesit', 'report', 'soport', 'tecnic']
despedida: ['adi', 'es', 'eso', 'graci', 'hast', 'lueg', 'tod']
saludo: ['buen', 'dias', 'hol', 'noch', 'salud', 'tard']
ayuda: ['ayud', 'ayudam', 'favor', 'necesit', 'orientam', 'por', 'pued']
documentacion: ['comprob', 'de', 'del', 'document', 'necesit', 'pag', 'que', 'requer', 'requisit', 'tramit']
procedimiento: ['com', 'nuev', 'oper

In [57]:
# Identificación de intenciones en consultas de Seguros del Norte.
# Cuando dos intenciones tienen el mismo puntaje máximo, se marca como ambigua.
def detectar_intencion(mensaje):
    if normalizar_texto(mensaje) == "salir":
        return "salir", 0
    _, stems_mensaje = tokenizar_y_stem(mensaje)
    stems_mensaje = set(stems_mensaje)
    puntajes = {}
    for intencion, stems_intencion in vocabulario_intenciones.items():
        coincidencias = stems_mensaje.intersection(stems_intencion)
        puntajes[intencion] = len(coincidencias)
    mejor_puntaje = max(puntajes.values())
    if mejor_puntaje == 0:
        return "desconocida", 0
    ganadoras = [k for k, v in puntajes.items() if v == mejor_puntaje]
    if len(ganadoras) > 1:
        return "ambigua", mejor_puntaje
    return ganadoras[0], mejor_puntaje

# Catálogo de temas operativos de Seguros del Norte.
catalogo_servicios = {
    "documentos": ["documentos", "documentacion", "comprobante"],
    "operaciones": ["operacion", "operaciones", "registro"],
    "archivos": ["adjuntar", "archivos", "subir"],
    "plataforma": ["plataforma", "pagina", "pantalla"],
}

def detectar_entidades(mensaje):
    texto = normalizar_texto(mensaje)
    entidades = {}
    for entidad, palabras_clave in catalogo_servicios.items():
        for palabra in palabras_clave:
            if palabra in texto.split():
                entidades["servicio"] = entidad
                return entidades
    entidades["servicio"] = "no especificado"
    return entidades

def generar_respuesta(mensaje):
    """
    Genera una respuesta considerando intención y entidades.
    """
    intencion, puntaje = detectar_intencion(mensaje)
    entidades = detectar_entidades(mensaje)

    if intencion == "salir":
        respuesta = "Sesión finalizada."
    elif intencion == "ambigua":
        respuesta = "Identifiqué varios temas. Consulta un solo tema por mensaje."
    elif intencion == "desconocida":
        respuesta = ("No entendí tu consulta. Puedes preguntarme sobre "
                     "documentación, procedimientos o uso de la plataforma.")
    else:
        respuesta = None

    if respuesta is not None:
        return {
            "intencion": intencion,
            "puntaje": puntaje,
            "entidades": entidades,
            "respuesta": respuesta
        }

    plantilla = random.choice(intenciones[intencion]["respuestas"])
    respuesta = plantilla.format(**entidades)

    return {
        "intencion": intencion,
        "puntaje": puntaje,
        "entidades": entidades,
        "respuesta": respuesta
    }

In [68]:
# Conversación interactiva con el asistente de Seguros del Norte.
def iniciar_chatbot():
    historial = []
    print("Chatbot: Hola, soy el asistente de Seguros del Norte.")
    print("Chatbot: Pregúntame por documentos, operaciones o uso de la plataforma.")
    print("Chatbot: Escribe 'salir' para terminar.\n")
    while True:
        mensaje_usuario = input("Usuario: ")
        if normalizar_texto(mensaje_usuario) == "salir":
            print("Chatbot: Sesión finalizada.")
            break
        resultado = generar_respuesta(mensaje_usuario)
        historial.append({
            "mensaje_usuario": mensaje_usuario,
            "intencion_detectada": resultado["intencion"],
            "puntaje": resultado["puntaje"],
            "entidades": resultado["entidades"],
            "respuesta": resultado["respuesta"]
        })
        print(f"Chatbot: {resultado['respuesta']}\n")
    return pd.DataFrame(historial)

historial_chat = iniciar_chatbot()
historial_chat


Chatbot: Hola, soy el asistente de Seguros del Norte.
Chatbot: Pregúntame por documentos, operaciones o uso de la plataforma.
Chatbot: Escribe 'salir' para terminar.

Usuario: 
Chatbot: No entendí tu consulta. Puedes preguntarme sobre documentación, procedimientos o uso de la plataforma.

Usuario: gracias
Chatbot: Gracias por tu consulta. Hasta pronto.

Usuario: salir
Chatbot: Sesión finalizada.


,mensaje_usuario,intencion_detectada,puntaje,entidades,respuesta
0,,desconocida,0,{'servicio': 'no especificado'},No entendí tu consulta. Puedes preguntarme sob...
1,gracias,despedida,1,{'servicio': 'no especificado'},Gracias por tu consulta. Hasta pronto.


In [59]:
# Comprobación automática de 12 entradas (adicional a las 10 conversaciones manuales).
casos_prueba = [
    {"mensaje": "hola", "intencion_esperada": "saludo"},
    {"mensaje": "necesito ayuda", "intencion_esperada": "ayuda"},
    {"mensaje": "que documentos necesito", "intencion_esperada": "documentacion"},
    {"mensaje": "como registro una operacion", "intencion_esperada": "procedimiento"},
    {"mensaje": "adjuntar documentos", "intencion_esperada": "adjuntar"},
    {"mensaje": "donde encuentro operaciones", "intencion_esperada": "ubicacion"},
    {"mensaje": "la pagina no carga", "intencion_esperada": "falla"},
    {"mensaje": "necesito soporte tecnico", "intencion_esperada": "soporte"},
    {"mensaje": "gracias", "intencion_esperada": "despedida"},
    {"mensaje": "zzzz1234", "intencion_esperada": "desconocida"},
    {"mensaje": "", "intencion_esperada": "desconocida"},
    {"mensaje": "salir", "intencion_esperada": "salir"}
]
resultados = []
for caso in casos_prueba:
    resultado = generar_respuesta(caso["mensaje"])
    resultados.append({
        "mensaje": caso["mensaje"],
        "intencion_esperada": caso["intencion_esperada"],
        "intencion_detectada": resultado["intencion"],
        "correcto": caso["intencion_esperada"] == resultado["intencion"],
        "respuesta": resultado["respuesta"]
    })
df_resultados = pd.DataFrame(resultados)
precision = df_resultados["correcto"].mean()
tasa_fallback = (df_resultados["intencion_detectada"] == "desconocida").mean()
print("Resultados de evaluación")
print("Precisión de intención:", f"{precision:.2%}")
print("Tasa de fallback:", f"{tasa_fallback:.2%}")
print("Aciertos:", int(df_resultados['correcto'].sum()), '/', len(df_resultados))
df_resultados


Resultados de evaluación
Precisión de intención: 100.00%
Tasa de fallback: 16.67%
Aciertos: 12 / 12


,mensaje,intencion_esperada,intencion_detectada,correcto,respuesta
0,hola,saludo,saludo,True,"Hola. Puedo orientarte sobre documentos, opera..."
1,necesito ayuda,ayuda,ayuda,True,"Claro. Indica si tu duda es sobre documentos, ..."
2,que documentos necesito,documentacion,documentacion,True,Los documentos dependen del trámite. Especific...
3,como registro una operacion,procedimiento,procedimiento,True,"Para registrar una operación, entra a Operacio..."
4,adjuntar documentos,adjuntar,adjuntar,True,"Para adjuntar archivos, abre el formulario y b..."
5,donde encuentro operaciones,ubicacion,ubicacion,True,Encontrarás Operaciones en el menú principal.
6,la pagina no carga,falla,falla,True,Actualiza la página y vuelve a intentarlo. Si ...
7,necesito soporte tecnico,soporte,soporte,True,El soporte humano atiende fallas técnicas. Si ...
8,gracias,despedida,despedida,True,Gracias por tu consulta. Hasta pronto.
9,zzzz1234,desconocida,desconocida,True,No entendí tu consulta. Puedes preguntarme sob...
